# Importação segura Google Sheets → Ploomes

Este notebook importa empresas e pessoas físicas a partir da planilha `Ploomes_Clientes`, usando a API do Ploomes diretamente.

Ele não depende de categoria `B2C` para decidir o tipo do cliente. A classificação é feita pela estrutura da linha:

- `Nome - Empresa` preenchido e `Nome - Pessoa` vazio: cria empresa sem pessoa.
- `Nome - Empresa` preenchido e `Nome - Pessoa` preenchido: cria empresa e cria pessoa vinculada à empresa.
- `Nome - Empresa` vazio e `Nome - Pessoa` preenchido: cria pessoa física sem empresa.

Pontos de segurança:

- Empresas são criadas como `TypeId=1`.
- Pessoas físicas são criadas como `TypeId=2`.
- Pessoas físicas sem empresa não recebem `CompanyId`, `LegalName`, `CNPJ` nem qualquer campo de empresa.
- O notebook começa em `DRY_RUN=True`, então primeiro ele só mostra o que faria.
- A chave da API deve ficar em `Colab Secrets` como `PLOOMES_API_KEY`, não no notebook.


## 1. Instalação, autenticação e configuração

No Colab, abra o painel `Secrets`, crie `PLOOMES_API_KEY`, cole sua chave e habilite o acesso ao notebook.

In [ ]:
!pip install -q gspread pandas requests

In [ ]:
import json
import time
import urllib.parse
from collections import Counter

import pandas as pd
import requests
from google.colab import auth, userdata
from google.auth import default
import gspread

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

SPREADSHEET_ID = "1cMzEfzHgn50QUjgNww8eqJIIgX6F-G8wskI8R1DzrnU"
WORKSHEET_NAME = "Clientes"
BASE_URL = "https://api2.ploomes.com"
PLOOMES_API_KEY = userdata.get("PLOOMES_API_KEY")

DRY_RUN = True
UPDATE_EXISTING = False
IMPORT_LINKED_PERSONS_IN_COMPANY_ROWS = True
REQUEST_SLEEP_SECONDS = 0.6
MAX_ROWS = None  # use 1 para testar só uma linha, None para lote completo

assert PLOOMES_API_KEY, "Configure PLOOMES_API_KEY no Colab Secrets."

HEADERS = {
    "User-Key": PLOOMES_API_KEY,
    "Accept": "application/json",
    "Content-Type": "application/json",
    "User-Agent": "Mozilla/5.0",
}


## 2. Ler a planilha

In [ ]:
sheet = gc.open_by_key(SPREADSHEET_ID).worksheet(WORKSHEET_NAME)
df = pd.DataFrame(sheet.get_all_records())
df = df.dropna(how="all")
print("linhas:", len(df))
print("colunas:")
for col in df.columns:
    print("-", col)
display(df.head())

## 3. Funções de limpeza e classificação

A classificação não usa `B2C`, marcadores, origem ou categoria. Ela usa apenas a presença dos nomes estruturais da sua planilha atual:

- `empresa_sem_pessoa`: tem `Razão Social - Empresa`, não tem `Nome - Pessoa`.
- `empresa_com_pessoa`: tem `Razão Social - Empresa` e `Nome - Pessoa`.
- `pessoa_fisica_sem_empresa`: não tem `Razão Social - Empresa`, tem `Nome - Pessoa`.
- `ignorar`: não tem nenhum dos dois nomes.

Isso permite gerar os três casos em uma única planilha mestre: empresas sem pessoas, empresas com pessoas e pessoas físicas B2C sem empresa.

In [ ]:
def clean(value):
    if pd.isna(value):
        return ""
    value = str(value).strip()
    if value.lower() in {"nan", "none", "null"}:
        return ""
    if value.endswith(".0") and value[:-2].isdigit():
        return value[:-2]
    return value

def pick(row, *names):
    for name in names:
        if name in row:
            value = clean(row.get(name))
            if value:
                return value
    return ""

def only_digits(value):
    return "".join(ch for ch in clean(value) if ch.isdigit())

def company_name(row):
    return pick(
        row,
        "Razão Social - Empresa",
        "Razao Social - Empresa",
        "Razão social - Empresa",
        "Razao social - Empresa",
        "Nome - Empresa",
        "Nome Empresa",
    )

def person_name(row):
    return pick(row, "Nome - Pessoa", "Nome Pessoa")

def row_kind(row):
    has_company = bool(company_name(row))
    has_person = bool(person_name(row))
    if has_company and has_person:
        return "empresa_com_pessoa"
    if has_company:
        return "empresa_sem_pessoa"
    if has_person:
        return "pessoa_fisica_sem_empresa"
    return "ignorar"

df["_kind"] = df.apply(row_kind, axis=1)
print(Counter(df["_kind"]))
preview_cols = [
    "_kind",
    "Razão Social - Empresa",
    "Nome - Pessoa",
    "E-mail - Pessoa",
    "Telefones - Pessoa",
    "CPF - Pessoa",
    "Origem - Empresa",
    "Marcadores - Empresa",
]
display(df[[c for c in preview_cols if c in df.columns]].head(80))


## 4. Funções da API do Ploomes

In [ ]:
def odata_escape(value):
    return clean(value).replace("'", "''")

def build_query(params):
    return "&".join(
        f"{key}={urllib.parse.quote(str(value), safe="()',$,/")}"
        for key, value in params.items()
    )

def api_request(method, path, params=None, payload=None):
    url = BASE_URL.rstrip("/") + path
    if params:
        url += "?" + build_query(params)
    response = requests.request(method, url, headers=HEADERS, json=payload, timeout=60)
    if response.status_code in (200, 201, 202, 204):
        if response.text.strip():
            return response.json()
        return {}
    raise RuntimeError(f"{method} {url} -> HTTP {response.status_code}: {response.text[:1000]}")

def fetch_all(path, params):
    items = []
    url = BASE_URL.rstrip("/") + path + "?" + build_query(params)
    while url:
        response = requests.get(url, headers=HEADERS, timeout=60)
        if response.status_code != 200:
            raise RuntimeError(f"GET {url} -> HTTP {response.status_code}: {response.text[:1000]}")
        data = response.json()
        items.extend(data.get("value", []))
        next_link = data.get("@odata.nextLink")
        url = urllib.parse.urljoin(BASE_URL.rstrip("/") + "/", next_link) if next_link else None
    return items

def first_contact(filter_expr):
    items = fetch_all("/Contacts", {"$filter": filter_expr, "$select": "Id,TypeId,Name,Email,Register", "$top": "1"})
    return items[0] if items else None

def find_company(payload):
    register = clean(payload.get("Register"))
    if register:
        found = first_contact(f"TypeId eq 1 and Register eq '{odata_escape(register)}'")
        if found:
            return found
    return first_contact(f"TypeId eq 1 and Name eq '{odata_escape(payload['Name'])}'")

def find_person(payload):
    email = clean(payload.get("Email"))
    register = clean(payload.get("Register"))
    if email:
        found = first_contact(f"TypeId eq 2 and Email eq '{odata_escape(email)}'")
        if found:
            return found
    if register:
        found = first_contact(f"TypeId eq 2 and Register eq '{odata_escape(register)}'")
        if found:
            return found
    return first_contact(f"TypeId eq 2 and Name eq '{odata_escape(payload['Name'])}'")

def load_origins():
    try:
        origins = fetch_all("/Contacts@Origins", {"$select": "Id,Name", "$top": "100"})
        return {clean(item.get("Name")).casefold(): item.get("Id") for item in origins if clean(item.get("Name"))}
    except Exception as exc:
        print("Aviso: não consegui carregar Contacts@Origins, OriginId será omitido.", exc)
        return {}

ORIGIN_BY_NAME = load_origins()
ORIGIN_BY_NAME

## 5. Montar payloads

Esta célula não envia nada. Ela só transforma linhas em payloads.

Mapeamento seguro para sua planilha atual:

- Empresa: usa `Razão Social - Empresa` como `Name` e `LegalName`.
- Pessoa física: usa `Nome - Pessoa`, `E-mail - Pessoa`, `CPF - Pessoa` e observações.
- Pessoa vinculada a empresa: só recebe `CompanyId` depois que a empresa for criada ou localizada.

Campos como `Segmento de atuação - Empresa`, `Estado - Empresa`, `Cidade - Empresa`, `Cargo - Pessoa`, `Departamento - Pessoa` e `Marcadores - Empresa` podem depender de IDs internos do Ploomes. Por segurança, este notebook não envia esses campos como texto cru na primeira versão.

In [ ]:
def origin_id(row):
    name = pick(row, "Origem - Empresa", "Origem - Pessoa", "Origem")
    return ORIGIN_BY_NAME.get(name.casefold()) if name else None

def joined_note(*parts):
    values = [clean(part) for part in parts if clean(part)]
    return "\n".join(values)

def build_company_payload(row):
    name = company_name(row)
    if not name:
        return None
    payload = {"TypeId": 1, "Name": name, "LegalName": name}
    cnpj = only_digits(pick(row, "CNPJ - Empresa", "CNPJ"))
    website = pick(row, "Site", "Site - Empresa")
    note = joined_note(
        pick(row, "Relação", "Relacao"),
        pick(row, "Observações", "Observacoes"),
    )
    oid = origin_id(row)
    if cnpj:
        payload["Register"] = cnpj
    if website:
        payload["Website"] = website
    if note:
        payload["Note"] = note
    if oid:
        payload["OriginId"] = oid
    return payload

def build_person_payload(row, company_id=None):
    name = person_name(row)
    if not name:
        return None
    payload = {"TypeId": 2, "Name": name}
    email = pick(row, "E-mail - Pessoa", "Email - Pessoa")
    cpf = only_digits(pick(row, "CPF - Pessoa", "CPF"))
    birthday = pick(row, "Data de nascimento - Pessoa", "Data nascimento - Pessoa")
    note = joined_note(
        f"Cargo: {pick(row, 'Cargo - Pessoa')}" if pick(row, "Cargo - Pessoa") else "",
        f"Departamento: {pick(row, 'Departamento - Pessoa')}" if pick(row, "Departamento - Pessoa") else "",
        pick(row, "Relação", "Relacao"),
        pick(row, "Observações", "Observacoes"),
    )
    oid = origin_id(row)
    if email:
        payload["Email"] = email
    if cpf:
        payload["Register"] = cpf
    if birthday:
        payload["Birthday"] = birthday
    if note:
        payload["Note"] = note
    if oid:
        payload["OriginId"] = oid
    if company_id:
        payload["CompanyId"] = company_id
    return payload

planned = []
for row_number, (_, row) in enumerate(df.iterrows(), start=2):
    kind = row["_kind"]
    if kind == "empresa_sem_pessoa":
        planned.append({"row": row_number, "kind": kind, "company": build_company_payload(row), "person": None})
    elif kind == "pessoa_fisica_sem_empresa":
        planned.append({"row": row_number, "kind": kind, "company": None, "person": build_person_payload(row)})
    elif kind == "empresa_com_pessoa":
        planned.append({
            "row": row_number,
            "kind": kind,
            "company": build_company_payload(row),
            "person": build_person_payload(row) if IMPORT_LINKED_PERSONS_IN_COMPANY_ROWS else None,
        })

if MAX_ROWS is not None:
    planned = planned[:MAX_ROWS]

print("payloads planejados:", len(planned))
print(Counter(item["kind"] for item in planned))
print(json.dumps(planned[:10], ensure_ascii=False, indent=2))


## 6. Travas contra o bug da Pluga

Esta célula falha se uma pessoa física estiver prestes a ser enviada com campo de empresa.

In [ ]:
bad_people = []
suspicious_companies = []
for item in planned:
    person = item.get("person")
    company = item.get("company")
    if item["kind"] == "pessoa_fisica" and person:
        forbidden = {"CompanyId", "LegalName", "CNPJ", "Company"}.intersection(person.keys())
        if forbidden:
            bad_people.append((item["row"], forbidden, person))
    if company and clean(company.get("Name")).casefold() in {"b2c", "empresa"}:
        suspicious_companies.append(item)

assert not bad_people, bad_people
if suspicious_companies:
    print("Atenção: há empresas com nome B2C/Empresa. Revise antes de enviar:")
    print(json.dumps(suspicious_companies, ensure_ascii=False, indent=2))
else:
    print("OK: nenhuma pessoa física será enviada com campo de empresa, e nenhuma empresa B2C/Empresa foi planejada.")

## 7. Executar importação

Primeiro rode com `DRY_RUN=True`. Depois teste com `MAX_ROWS=1` e `DRY_RUN=False`. Só rode o lote inteiro depois de conferir no Ploomes.

In [ ]:
def create_or_update_contact(payload, finder):
    existing = finder(payload)
    if existing:
        if not UPDATE_EXISTING:
            return {"action": "skipped_existing", "existing": existing, "payload": payload}
        if DRY_RUN:
            return {"action": "dry_run_update", "existing": existing, "payload": payload}
        data = api_request("PATCH", f"/Contacts({existing['Id']})", payload=payload)
        return {"action": "updated", "existing": existing, "response": data, "payload": payload}
    if DRY_RUN:
        return {"action": "dry_run_create", "payload": payload}
    data = api_request("POST", "/Contacts", payload=payload)
    return {"action": "created", "response": data, "payload": payload}

results = []
errors = []

for item in planned:
    try:
        company_result = None
        company_id = None
        if item.get("company"):
            company_result = create_or_update_contact(item["company"], find_company)
            company_id = (
                company_result.get("response", {}).get("Id")
                or company_result.get("existing", {}).get("Id")
            )

        person_payload = item.get("person")
        person_result = None
        if person_payload:
            if item["kind"] == "empresa_com_pessoa" and company_id:
                person_payload = dict(person_payload)
                person_payload["CompanyId"] = company_id
            person_result = create_or_update_contact(person_payload, find_person)

        results.append({"row": item["row"], "kind": item["kind"], "company": company_result, "person": person_result})
        print("OK", item["row"], item["kind"])
    except Exception as exc:
        errors.append({"row": item["row"], "kind": item["kind"], "error": str(exc), "item": item})
        print("ERRO", item["row"], item["kind"], exc)
    time.sleep(REQUEST_SLEEP_SECONDS)

print("resultados:", Counter(r.get("company", {}).get("action") for r in results if r.get("company")))
print("pessoas:", Counter(r.get("person", {}).get("action") for r in results if r.get("person")))
print("erros:", len(errors))
errors[:3]

## 8. Verificação rápida no Ploomes

In [ ]:
def count_contacts(type_id):
    items = fetch_all("/Contacts", {"$filter": f"TypeId eq {type_id}", "$select": "Id", "$top": "100"})
    return len(items)

print("empresas TypeId=1:", count_contacts(1))
print("pessoas TypeId=2:", count_contacts(2))
print("empresas chamadas B2C:", fetch_all("/Contacts", {"$filter": "TypeId eq 1 and Name eq 'B2C'", "$select": "Id,Name,LegalName", "$top": "20"}))
print("empresas chamadas Empresa:", fetch_all("/Contacts", {"$filter": "TypeId eq 1 and Name eq 'Empresa'", "$select": "Id,Name,LegalName", "$top": "20"}))